# Evaluate Extracted Experimental Layer Images

This notebook evaluates the extracted background-corrected layer images with both models:

- fully data-driven ResNet
- crystallographic constrained ResNet

Layers `0..5` use `n = (1, -1, -1)`. The remaining layers use `n = (-1, 1, 1)`. The true Burgers vector for every layer is `b = [-1, 0, -1]`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from bv_repro.data import load_class_to_burgers
from bv_repro.models import ResNet18Gray, ResNetCrystal
from bv_repro.training import (
    BenchmarkConfig,
    apply_crystal_mask,
    get_device,
    make_valid_mask_from_normal,
)


## Configure paths and labels

In [ ]:
DATA_DIR = Path("/data/projects/engage_id03/bv_paper/edf_layer_background_corrected_integrated")
ARTIFACT_DIR = Path("benchmark_artifacts")

TRUE_BURGERS = np.array([-1, 0, -1], dtype=np.int32)
NORMAL_FIRST = (1, -1, -1)   # layers 0..5
NORMAL_REST = (-1, 1, 1)     # layers 6..end

# The extracted images were already resized/normalized when saved.
# Use "as_saved" first. Use "log" only if you intentionally want log1p contrast before inference.
PREPROCESS_MODE = "as_saved"  # "as_saved" or "log"

config = BenchmarkConfig(
    data_path="/data/projects/engage_id03/int_fcc_002/",
    artifact_dir=str(ARTIFACT_DIR),
)

print("DATA_DIR exists:", DATA_DIR.exists())
print("ARTIFACT_DIR exists:", ARTIFACT_DIR.exists())


## Load extracted layer images

In [ ]:
image_paths = sorted(DATA_DIR.glob("layer_*_integrated_from_4th_frame.npy"))
if not image_paths:
    raise FileNotFoundError(f"No layer integrated images found in {DATA_DIR}")

def layer_number(path):
    # layer_00_integrated_from_4th_frame.npy -> 0
    return int(path.name.split("_")[1])

image_paths = sorted(image_paths, key=layer_number)
layer_numbers = [layer_number(p) for p in image_paths]
normals = [NORMAL_FIRST if n <= 5 else NORMAL_REST for n in layer_numbers]

print("found images:", len(image_paths))
for path, normal in zip(image_paths, normals):
    print(f"layer {layer_number(path):02d}: {path.name}, n={normal}")


## Load models

In [ ]:
device = get_device()
class_to_burgers = load_class_to_burgers(config=config, artifact_dir=ARTIFACT_DIR)
num_classes = len(class_to_burgers)

data_model = ResNet18Gray(num_classes=num_classes, pretrained=False).to(device)
crystal_model = ResNetCrystal(num_classes=num_classes, pretrained=False).to(device)

data_model.load_state_dict(torch.load(ARTIFACT_DIR / "data_driven_resnet18.pt", map_location=device))
crystal_model.load_state_dict(torch.load(ARTIFACT_DIR / "crystallographic_resnet18.pt", map_location=device))

data_model.eval()
crystal_model.eval()

print("device:", device)
print("classes:")
for i, b in enumerate(class_to_burgers):
    print(i, [int(x) for x in b])


## Predict probabilities for each layer

In [ ]:
def preprocess_extracted_image(path, mode="as_saved"):
    img = np.load(path).astype(np.float32)
    if img.ndim == 2:
        img = np.expand_dims(img, axis=0)

    if mode == "log":
        img = np.log1p(np.clip(img, a_min=0.0, a_max=None))
    elif mode != "as_saved":
        raise ValueError("mode must be 'as_saved' or 'log'")

    t = torch.tensor(img, dtype=torch.float32)
    t = t - t.min()
    if t.max() > 0:
        t = t / t.max()
    return t

def burgers_label(b):
    b = [int(x) for x in b]
    return f"[{b[0]}, {b[1]}, {b[2]}]"

def class_for_burgers(target_b, class_to_burgers):
    target_b = np.asarray(target_b, dtype=np.int32)
    for idx, b in enumerate(class_to_burgers):
        b = np.asarray(b, dtype=np.int32)
        if np.array_equal(b, target_b) or np.array_equal(-b, target_b):
            return idx
    raise ValueError(f"Could not find class for b={target_b.tolist()}")

true_class = class_for_burgers(TRUE_BURGERS, class_to_burgers)
class_labels = [burgers_label(b) for b in class_to_burgers]

records = []

with torch.no_grad():
    for path, layer_idx, normal in zip(image_paths, layer_numbers, normals):
        x = preprocess_extracted_image(path, mode=PREPROCESS_MODE).unsqueeze(0).to(device)

        data_logits = data_model(x)
        data_probs = F.softmax(data_logits, dim=1).squeeze(0).cpu().numpy()

        crystal_logits = crystal_model(x)
        valid_mask = make_valid_mask_from_normal(class_to_burgers, normal).unsqueeze(0).to(device)
        crystal_logits = apply_crystal_mask(crystal_logits, valid_mask)
        crystal_probs = F.softmax(crystal_logits, dim=1).squeeze(0).cpu().numpy()

        records.append({
            "path": path,
            "layer": layer_idx,
            "normal": normal,
            "data_probs": data_probs,
            "crystal_probs": crystal_probs,
            "data_pred": int(np.argmax(data_probs)),
            "crystal_pred": int(np.argmax(crystal_probs)),
        })

print("true b:", TRUE_BURGERS.tolist(), "true class:", true_class, class_labels[true_class])
for r in records:
    print(
        f"layer {r['layer']:02d} n={r['normal']} | "
        f"data={class_labels[r['data_pred']]} p={r['data_probs'][r['data_pred']]:.3f} | "
        f"crystal={class_labels[r['crystal_pred']]} p={r['crystal_probs'][r['crystal_pred']]:.3f}"
    )


## Plot probability for each model and layer

In [ ]:
data_prob_matrix = np.stack([r["data_probs"] for r in records], axis=0)
crystal_prob_matrix = np.stack([r["crystal_probs"] for r in records], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

for ax, matrix, title in [
    (axes[0], data_prob_matrix, "Data-driven"),
    (axes[1], crystal_prob_matrix, "Crystallographic constrained"),
]:
    im = ax.imshow(matrix.T, aspect="auto", vmin=0, vmax=1, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("Layer")
    ax.set_xticks(np.arange(len(layer_numbers)))
    ax.set_xticklabels([str(n) for n in layer_numbers])
    ax.set_yticks(np.arange(num_classes))
    ax.set_yticklabels(class_labels)
    ax.axhline(true_class, color="red", linewidth=2, linestyle="--", alpha=0.8)

axes[0].set_ylabel("Burgers vector class")
fig.colorbar(im, ax=axes, label="Probability", shrink=0.9)
plt.tight_layout()
plt.savefig("experimental_layer_model_probabilities.png", dpi=300, bbox_inches="tight")
plt.show()


## Mean probability distribution across layers

In [ ]:
mean_data_probs = data_prob_matrix.mean(axis=0)
mean_crystal_probs = crystal_prob_matrix.mean(axis=0)

x = np.arange(num_classes)
width = 0.38

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(x - width / 2, mean_data_probs, width, label="Data-driven", color="tab:blue", alpha=0.85)
ax.bar(x + width / 2, mean_crystal_probs, width, label="Crystallographic constrained", color="tab:orange", alpha=0.85)
ax.axvline(true_class, color="red", linestyle="--", linewidth=2, label="True b class")
ax.set_xticks(x)
ax.set_xticklabels(class_labels, rotation=35, ha="right")
ax.set_xlabel("Burgers vector class")
ax.set_ylabel("Mean probability across layers")
ax.set_ylim(0, 1.05)
ax.grid(True, axis="y", alpha=0.3)
ax.legend(frameon=True)
plt.tight_layout()
plt.savefig("experimental_layer_mean_probability_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


## Confusion matrix

In [ ]:
def confusion_from_predictions(preds, true_class, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=float)
    for pred in preds:
        cm[true_class, pred] += 1
    return cm


def row_percent_confusion(cm):
    row_sums = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums > 0) * 100.0


data_preds = np.array([r["data_pred"] for r in records], dtype=int)
crystal_preds = np.array([r["crystal_pred"] for r in records], dtype=int)

cm_data = confusion_from_predictions(data_preds, true_class, num_classes)
cm_crystal = confusion_from_predictions(crystal_preds, true_class, num_classes)

cm_data_pct = row_percent_confusion(cm_data)
cm_crystal_pct = row_percent_confusion(cm_crystal)

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), constrained_layout=True)

for ax, cm_pct, title in [
    (axes[0], cm_data_pct, "Data-driven"),
    (axes[1], cm_crystal_pct, "Crystallographic constrained"),
]:
    im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100, aspect="equal")
    ax.set_title(title, pad=10)
    ax.set_xlabel("Predicted Burgers vector")
    ax.set_ylabel("True Burgers vector")
    ax.set_xticks(np.arange(num_classes))
    ax.set_xticklabels(class_labels, rotation=35, ha="right")
    ax.set_yticks(np.arange(num_classes))
    ax.set_yticklabels(class_labels)

    ax.set_xticks(np.arange(-0.5, num_classes, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, num_classes, 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    for i in range(num_classes):
        for j in range(num_classes):
            value = cm_pct[i, j]
            if value > 0:
                text_color = "white" if value >= 50 else "black"
                ax.text(
                    j,
                    i,
                    f"{value:.1f}%",
                    ha="center",
                    va="center",
                    color=text_color,
                    fontsize=11,
                    fontweight="bold",
                )

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.82, pad=0.02)
cbar.set_label("Prediction percentage (%)")

plt.savefig("experimental_layer_confusion_matrices_percent.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Data-driven accuracy: {(data_preds == true_class).mean() * 100:.1f}%")
print(f"Crystallographic accuracy: {(crystal_preds == true_class).mean() * 100:.1f}%")
